# AmsiNet - Step 16: cheaper flat baselines at ten classes (A100, about 20-30 minutes)
**Why.** The paper needs a flat classifier whose cost is comparable to AmsiNet's, not only a ViT-B/16. This step trains two flat models on the same word crops, split and seeds as Step 2, with the same ten classes:
* **ViT-S/16** (about 9 GFLOPs per word, roughly 10x the AmsiNet cascade per image)
* **EfficientNet-B0** (about 0.8 GFLOPs per word, close to the cascade's cost per image)

Five seeds each, 8 epochs, best epoch chosen on validation macro-F1, probabilities saved to `agents/flat_vits_seed{k}_*` and `agents/flat_effb0_seed{k}_*`, in the same format as the other agents. Needs `crops224.tar`, `crops_meta.csv` and `bstd_index.json` from Step 2. Afterwards run Step 17 on a CPU runtime and paste its summary table.

In [ ]:
# 1. Setup
from google.colab import drive
import os, subprocess, sys
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
try: import timm
except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=True); import timm
import json, glob, random, time, math, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from scipy import stats
from scipy.special import softmax
from sklearn.metrics import f1_score
ROOT, LOCAL = '/content/drive/MyDrive/amsinet', '/content/bstd'; os.makedirs(LOCAL, exist_ok=True)
CFG = dict(SEEDS=[0, 1, 2, 3, 4], EPOCHS=8, BATCH=64, WORKERS=8)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SCRIPTS, INDEX = D['scripts'], D['index']; IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); META['y'] = META.script.map(SIDX).astype(int)
print(C, 'scripts:', SCRIPTS); print(META.groupby('split').size().to_dict())
CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR): print('restoring crops'); subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)
assert os.path.exists(META.path.iloc[0]), 'crops not found on disk: ' + META.path.iloc[0]
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## 2. Training utilities

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
DEV = 'cuda'; FILL = (124, 116, 104)
class Crops(Dataset):
    def __init__(self, paths, labels, tf): self.p, self.y, self.tf = list(paths), list(labels), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]
def tfs(model):
    cfg = timm.data.resolve_data_config({}, model=model); norm = T.Normalize(cfg['mean'], cfg['std'])
    return (T.Compose([T.RandomAffine(5, translate=(0.03, 0.03), scale=(0.9, 1.1), fill=FILL), T.ColorJitter(0.3, 0.3, 0.2), T.ToTensor(), norm]), T.Compose([T.ToTensor(), norm]))
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def predict(model, df, tf):
    model.eval(); out = []
    for x, _ in DataLoader(Crops(df.path, df.y, tf), batch_size=256, num_workers=CFG['WORKERS']):
        with torch.autocast('cuda', dtype=torch.float16): out.append(model(x.to(DEV)).float().softmax(1).cpu())
    return torch.cat(out).numpy()
def mf1(P, y): return float(f1_score(y, P.argmax(1), average='macro', labels=list(range(C)), zero_division=0))
def train(tag, model_name, tr, va, epochs, lr, seed):
    # trains on dataframe tr, selects the epoch by macro-F1 on dataframe va, saves probabilities for ALL val and test words
    files = [f'{ROOT}/agents/{tag}_seed{seed}_{sp}_probs.npy' for sp in ('val', 'test')]
    if all(os.path.exists(f) for f in files): print('skip (done):', tag, seed); return
    set_seed(seed); model = timm.create_model(model_name, pretrained=True, num_classes=C).to(DEV); ttf, etf = tfs(model)
    L = DataLoader(Crops(tr.path, tr.y, ttf), batch_size=CFG['BATCH'], shuffle=True, num_workers=CFG['WORKERS'], pin_memory=True, drop_last=True)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total = epochs * len(L); warm = max(1, int(0.05 * total)); sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda'); best, state = -1, None
    for ep in range(epochs):
        model.train(); t0 = time.time(); rl = 0
        for x, y in L:
            x, y = x.to(DEV, non_blocking=True), torch.as_tensor(y).to(DEV)
            with torch.autocast('cuda', dtype=torch.float16): loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); rl += loss.item()
        Pv = predict(model, va, etf); f = mf1(Pv, va.y.values)
        print(f'{tag} seed{seed} ep{ep + 1}: loss {rl / len(L):.4f} val acc {(Pv.argmax(1) == va.y.values).mean():.4f} macro-F1 {f:.4f} ({time.time() - t0:.0f}s)')
        if f > best: best = f; state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state)
    for sp in ('val', 'test'):
        d = META[META.split == sp]; P = predict(model, d, etf); np.save(f'{ROOT}/agents/{tag}_seed{seed}_{sp}_probs.npy', P.astype(np.float16)); pd.DataFrame({'path': d.path.values}).to_csv(f'{ROOT}/agents/{tag}_seed{seed}_{sp}_paths.csv', index=False)
    print(f'done {tag} seed{seed}: best val macro-F1 {best:.4f}'); del model; torch.cuda.empty_cache()

## 3. Train

In [ ]:
TRN, VAL = META[META.split == 'train'], META[META.split == 'val']
RUNS = [('flat_effb0', 'efficientnet_b0.ra_in1k', 5e-4), ('flat_vits', 'vit_small_patch16_224.augreg_in21k', 1e-4)]
for seed in CFG['SEEDS']:
    for tag, model_name, lr in RUNS: train(tag, model_name, TRN, VAL, CFG['EPOCHS'], lr, seed)
print('training done')

## 4. Quick test-set check (not for model selection)

In [ ]:
TE = META[META.split == 'test']; yt = TE.y.values
for tag, _, _ in RUNS:
    a, f = [], []
    for seed in CFG['SEEDS']:
        P = np.load(f'{ROOT}/agents/{tag}_seed{seed}_test_probs.npy').astype(np.float32); a.append(100 * (P.argmax(1) == yt).mean()); f.append(100 * mf1(P, yt))
    print(f'{tag}: accuracy {np.mean(a):.2f} macro-F1 {np.mean(f):.2f} (mean of {len(a)} seeds)')
print('Now disconnect this runtime and run Step 17 on a CPU runtime.')